# Rule-based vs ML: comparison on the same test set

We compare the **full** rule-based engine (`fraud_checks/services.py`,
all 6 signals) against the ML model on the same test set.
Rules 1–6 cover: account_blocked, insufficient_balance,
limit_exceeded (daily > 200k), high_amount (> 100k),
new_account (< 7d), high_frequency (> 10tx/h).

In [2]:
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", ".."))
from ml.synthetic.build_dataset import generate_accounts

DATASET_PATH = "../datasets/synthetic_v2.csv"
MODEL_PATH = "../models/fraud_model_v2.pkl"
FEATURE_COLS = ["amount", "account_age_days", "tx_last_hour", "transaction_hour",
              "receiver_tx_count_24h", "sender_daily_amount_sum",
              "amount_to_balance_ratio", "days_since_last_tx"]

df = pd.read_csv(DATASET_PATH, parse_dates=["created_at"])
df = df.sort_values("created_at").reset_index(drop=True)

# Re-generate accounts (deterministic, SEED=42) for balance info
from ml.synthetic import config
accounts_df = generate_accounts(config.N_ACCOUNTS, np.random.default_rng(config.SEED))

# same time-based split as in train_model.py (train_ratio=0.8)
split_idx = int(len(df) * 0.8)
test_df = df.iloc[split_idx:].copy()

model = joblib.load(MODEL_PATH)
print(f"Test set: {len(test_df)} rows ({test_df["is_fraud"].mean():.2%} fraud)")

Test set: 4000 rows (6.33% fraud)


## Step 1 — ML model predictions

In [3]:
X_test = test_df[FEATURE_COLS]
test_df["ml_proba"] = model.predict_proba(X_test)[:, 1]
test_df["ml_pred"] = (test_df["ml_proba"] >= 0.5).astype(int)

## Step 2 — Rule-based engine (full set)

Logic ported from `fraud_checks/services.py` `calculate_risk()`.
All 6 signals are computed from DataFrame columns:
- account_blocked: is_blocked (False for all synthetic)
- insufficient_balance: amount > sender_balance
- limit_exceeded: sender_daily_amount_sum + amount > 200k
- high_amount: amount > 100k
- high_frequency: tx_last_hour > 10
- new_account: account_age_days < 7

In [4]:
def rule_based_full(row):
    risk_score = 0
    reasons = []
    if row["account_age_days"] < 7:
        risk_score += 20
        reasons.append("new_account")
    if row["amount"] > 100000:
        risk_score += 40
        reasons.append("high_amount")
    if row["tx_last_hour"] > 10:
        risk_score += 30
        reasons.append("high_frequency")
    if row["sender_daily_amount_sum"] + row["amount"] > 200000:
        risk_score += 60
        reasons.append("limit_exceeded")
    if row["amount"] > row["sender_balance_before"]:
        risk_score += 80
        reasons.append("insufficient_balance")
    # account_blocked: is_blocked is False for all synthetic, never fires

    if risk_score >= 60:
        decision = "BLOCKED"
    elif risk_score >= 30:
        decision = "REVIEW"
    else:
        decision = "APPROVED"
    return risk_score, decision, reasons

# Add sender balance column for insufficient_balance rule
balance_map = accounts_df.set_index("account_id")["balance"]
test_df["sender_balance_before"] = test_df["sender_account_id"].map(balance_map)

results = test_df.apply(rule_based_full, axis=1, result_type="expand")
test_df["rule_risk_score"] = results[0]
test_df["rule_decision"] = results[1]
test_df["rule_reasons"] = results[2]
# REVIEW and BLOCKED count as "system flagged something" -> 1
test_df["rule_pred"] = test_df["rule_decision"].isin(["BLOCKED", "REVIEW"]).astype(int)

## Step 3 — Metric comparison (full rules vs ML)

In [5]:
print("=== Rule-based (subset) ===\n")
print(classification_report(test_df["is_fraud"], test_df["rule_pred"],
                              target_names=["normal", "fraud"]))

print("\n=== ML model ===\n")
print(classification_report(test_df["is_fraud"], test_df["ml_pred"],
                              target_names=["normal", "fraud"]))

=== Rule-based (subset) ===

              precision    recall  f1-score   support

      normal       0.97      0.25      0.39      3747
       fraud       0.08      0.91      0.14       253

    accuracy                           0.29      4000
   macro avg       0.52      0.58      0.27      4000
weighted avg       0.92      0.29      0.38      4000


=== ML model ===

              precision    recall  f1-score   support

      normal       0.96      0.99      0.97      3747
       fraud       0.67      0.42      0.51       253

    accuracy                           0.95      4000
   macro avg       0.81      0.70      0.74      4000
weighted avg       0.94      0.95      0.94      4000



## Step 4 — Where they disagree

The most informative part: examining specific cases of disagreement.

In [6]:
# ML catches, rule misses (ML better here)
ml_catches_rule_misses = test_df[
    (test_df["is_fraud"] == 1) &
    (test_df["rule_pred"] == 0) &
    (test_df["ml_pred"] == 1)
]
print(f"ML caught fraud that rules missed: {len(ml_catches_rule_misses)}")
disp_cols = ["amount", "account_age_days", "tx_last_hour",
             "receiver_tx_count_24h", "sender_daily_amount_sum",
             "amount_to_balance_ratio", "days_since_last_tx",
             "fraud_pattern"]

print(ml_catches_rule_misses[disp_cols].head(10))

# Rule catches, ML misses (rule better here)
rule_catches_ml_misses = test_df[
    (test_df["is_fraud"] == 1) &
    (test_df["rule_pred"] == 1) &
    (test_df["ml_pred"] == 0)
]
print(f"\nRules caught fraud that ML missed: {len(rule_catches_ml_misses)}")
print(rule_catches_ml_misses[disp_cols].head(10))

# Both missed (most dangerous — neither system reacted)
both_miss = test_df[
    (test_df["is_fraud"] == 1) &
    (test_df["rule_pred"] == 0) &
    (test_df["ml_pred"] == 0)
]
print(f"\nBoth missed: {len(both_miss)}")
print(both_miss[disp_cols].head(10))

ML caught fraud that rules missed: 13
         amount  account_age_days  tx_last_hour  receiver_tx_count_24h  \
16116  10864.77               526             0                      0   
16196   3748.73               350             0                      0   
16941   1619.24                99             0                      0   
17356    805.02               695             0                      6   
17357   8291.13               117             0                      7   
17683    380.08               250             0                      0   
18214    182.99                 2             0                      0   
18548   5900.70                 0             0                      0   
18793   1160.66                75             0                      0   
18918   5430.61                 0             0                      0   

       sender_daily_amount_sum  amount_to_balance_ratio  days_since_last_tx  \
16116                      0.0                   0.9211           33

## Conclusions

- Full rule engine now covers all 6 signals from `calculate_risk()`
- 4 new fraud patterns (mule, structuring, balance_drain, dormant_reactivation) added to ML dataset
- 4 new features (receiver_tx_count_24h, sender_daily_amount_sum, amount_to_balance_ratio, days_since_last_tx) added to ML training
- Expected: `receiver_tx_count_24h` and `amount_to_balance_ratio` show high feature importance, closing the "both missed" gap
- The 92 "both missed" cases from v1 should significantly shrink since new patterns target previously uncovered rule surfaces